## Laxmi Kanth Oruganti
### MSCS-634 : Advanced Big Data and Data Mining
### Lab 2: Classification Using KNN and RNN Algorithms

In [2]:
# Load necessary packages for this assignment
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine

## Step 1: Load and Prepare the Dataset

In [17]:
# Load Wine Dataset from sklearn
wine_data = load_wine()
wine_df = pd.DataFrame(data=np.c_[wine_data['data'], wine_data['target']],
                       columns=wine_data['feature_names'] + ['target'])
# wine_data
wine_df

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,2.0
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,2.0
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,2.0
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,2.0


In [20]:
# Data exploration to review feature details and class distribution
wine_df.info()

wine_df.head(100)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  targe

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,12.47,1.52,2.20,19.0,162.0,2.50,2.27,0.32,3.28,2.60,1.16,2.63,937.0,1.0
96,11.81,2.12,2.74,21.5,134.0,1.60,0.99,0.14,1.56,2.50,0.95,2.26,625.0,1.0
97,12.29,1.41,1.98,16.0,85.0,2.55,2.50,0.29,1.77,2.90,1.23,2.74,428.0,1.0
98,12.37,1.07,2.10,18.5,88.0,3.52,3.75,0.24,1.95,4.50,1.04,2.77,660.0,1.0


In [35]:
# Split the dataset into 80% training and 20% testing sets.

from sklearn.model_selection import train_test_split

# Seperate Features and Target into different sets
# Features
x = wine_df.drop('target', axis=1)
# Target variables
y = wine_df['target']

# Split data into 80% training and 20% test.
# random_state = 42 ensures data splits the same every time it's run
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Verify the sizes of training and test data
print(f"Training set size: {len(x_train)} samples")
print(f"Test set size: {len(x_test)} samples")

Training set size: 142 samples
Test set size: 36 samples


## Step 2: Implement K-Nearest Neighbors (KNN)

In [49]:
# Implement the KNN Classifier using values of k such as 1, 5, 11, 15, and 21
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

k_neighbors = [1, 5, 11, 15, 21]

# Interate through the k_neighbors array to train the model with each k value and it's prediction accuracy
for k in k_neighbors:
    knn = KNeighborsClassifier(n_neighbors=k)

    # Training the model with the train data
    knn.fit(x_train, y_train)

    # Evaluate the model on the test data and get the prediction
    y_pred = knn.predict(x_test)

    # Calculate the accuracy score for each k value
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Model accuracy for k={k}: {accuracy:.2%}")


Model accuracy for k=1: 77.78%
Model accuracy for k=5: 72.22%
Model accuracy for k=11: 75.00%
Model accuracy for k=15: 75.00%
Model accuracy for k=21: 77.78%


## Step 3: Implement Radius Neighbors (RNN)

In [50]:
# Implement the RNN Classifier using radius values such as 350, 400, 450, 500, 550, and 600.
from sklearn.neighbors import RadiusNeighborsClassifier

radius_neighbors = [350,  400, 450, 500, 550, 600]

for radius in radius_neighbors:
    rnn = RadiusNeighborsClassifier(radius=radius)
    # Train the model with RNN algorithm
    rnn.fit(x_train, y_train)

    # Find the prediction for each r value
    y_pred = rnn.predict(x_test)

    # Calculate accuracy score
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Model accuracy for radius={radius}: {accuracy:.2%}")


Model accuracy for radius=350: 75.00%
Model accuracy for radius=400: 72.22%
Model accuracy for radius=450: 72.22%
Model accuracy for radius=500: 72.22%
Model accuracy for radius=550: 72.22%
Model accuracy for radius=600: 72.22%
